## 1️⃣ Configuration PySpark optimisée

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, sum as spark_sum, count as spark_count, 
    broadcast, when, isnan, isnull, regexp_replace,
    to_date, date_format, round as spark_round
)
from pyspark.sql.types import *
import pandas as pd
import os

# Configuration PySpark optimisée pour FreshKart
spark = SparkSession.builder \
    .appName("FreshKart Migration") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark Session créée - Version: {spark.version}")
print(f"🎯 Spark UI disponible sur: http://localhost:4040")

## 2️⃣ Chargement intelligent des données

In [ ]:
# 📂 Chemins des données (Docker)
base_path = "/workspace/Brief_Starter_Pack"

customers_path = f"{base_path}/data/march-input/customers.csv"
orders_pattern = f"{base_path}/data/march-input/orders_2025-03-*.json"
refunds_path = f"{base_path}/data/march-input/refunds.csv"

print("📂 Vérification des fichiers...")
print(f"Base path: {base_path}")
print(f"Customers: {os.path.exists(customers_path)}")
print(f"Refunds: {os.path.exists(refunds_path)}")

# Compter les fichiers JSON
json_files = glob.glob(f"{base_path}/data/march-input/orders_2025-03-*.json")
print(f"Fichiers JSON trouvés: {len(json_files)}")

if len(json_files) == 0:
    print("❌ Aucun fichier JSON trouvé")
    print(f"🔍 Contenu du dossier march-input:")
    march_input_path = f"{base_path}/data/march-input"
    if os.path.exists(march_input_path):
        files = os.listdir(march_input_path)
        for file in files[:5]:  # Afficher les 5 premiers
            print(f"  - {file}")
        if len(files) > 5:
            print(f"  ... et {len(files) - 5} autres fichiers")
    else:
        print(f"❌ Dossier march-input non trouvé: {march_input_path}")
else:
    print(f"✅ {len(json_files)} fichiers JSON disponibles")


### 🚀 Chargement PySpark vs Pandas

In [ ]:
# Démarrer le chronomètre
start_time = time.time()

# ⚡ PYSPARK : Chargement intelligent - UNE SEULE LIGNE pour 31 fichiers !
print("🚀 Chargement PySpark : 31 fichiers JSON en parallèle...")
orders_df = spark.read \
    .option("multiline", "true") \
    .json(orders_pattern)

# Chargement CSV avec schema inference
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(customers_path)

refunds_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(refunds_path)

loading_time = time.time() - start_time
print(f"⚡ Chargement terminé en {loading_time:.2f}s")

# Aperçu rapide des données
print(f"\n📊 Aperçu des données:")
print(f"- Commandes: {orders_df.count():,} lignes")
print(f"- Clients: {customers_df.count():,} lignes")
print(f"- Remboursements: {refunds_df.count():,} lignes")

## 3️⃣ Exploration des schemas

In [ ]:
# Schema des commandes JSON
print("📋 Schema commandes (JSON):")
orders_df.printSchema()

print("\n" + "="*50)
print("📋 Schema clients (CSV):")
customers_df.printSchema()

print("\n" + "="*50)
print("📋 Schema remboursements (CSV):")
refunds_df.printSchema()

## 4️⃣ Nettoyage et transformation des données

In [ ]:
# 🧹 NETTOYAGE CLIENTS
print("🧹 Nettoyage des clients...")
clean_customers = customers_df.filter(
    col("status") == "active"
).cache()  # Cache car réutilisé

print(f"Clients actifs: {clean_customers.count():,}/{customers_df.count():,}")

# 🧹 NETTOYAGE COMMANDES
print("\n🧹 Nettoyage des commandes...")
clean_orders = orders_df.filter(
    col("status") == "paid"
).cache()  # Cache car réutilisé

print(f"Commandes payées: {clean_orders.count():,}/{orders_df.count():,}")

# 🧹 NETTOYAGE REMBOURSEMENTS
print("\n🧹 Nettoyage des remboursements...")
clean_refunds = refunds_df \
    .withColumn("amount", 
        when(col("amount").rlike("^[0-9.,]+$"), 
             regexp_replace(col("amount"), ",", ".").cast("double")) \
        .otherwise(0.0)) \
    .filter(col("amount") > 0) \
    .cache()  # Cache car réutilisé

print(f"Remboursements valides: {clean_refunds.count():,}/{refunds_df.count():,}")

## 5️⃣ Explosion JSON native - Le pouvoir de PySpark ! 💥

In [ ]:
# 💥 EXPLOSION JSON - Fonctionnalité native PySpark
print("💥 Explosion des items JSON...")

# Avec PySpark : explosion native optimisée
orders_items = clean_orders \
    .withColumn("item", explode(col("items"))) \
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_date"),
        col("channel"),
        col("item.sku").alias("item_sku"),
        col("item.qty").alias("item_qty"),
        col("item.unit_price").alias("item_unit_price")
    ) \
    .filter(col("item_unit_price") > 0) \
    .cache()  # Cache car base pour calculs

print(f"Items explosés: {orders_items.count():,}")

# Aperçu des données explosées
print("\n📊 Aperçu des items:")
orders_items.show(5, truncate=False)

## 6️⃣ Jointures optimisées avec Broadcast

In [ ]:
# 🔗 JOINTURES OPTIMISÉES
print("🔗 Jointures avec broadcast...")

# Broadcast des petites tables (clients) pour optimiser les jointures
orders_with_customers = orders_items.join(
    broadcast(clean_customers),  # Broadcast = optimisation jointure
    "customer_id",
    "inner"
).select(
    col("order_id"),
    col("customer_id"),
    col("order_date"),
    col("channel"),
    col("city"),
    col("item_sku"),
    col("item_qty"),
    col("item_unit_price"),
    (col("item_qty") * col("item_unit_price")).alias("line_revenue")
).cache()

print(f"Commandes + clients: {orders_with_customers.count():,}")

# Jointure avec remboursements (left join)
final_data = orders_with_customers.join(
    clean_refunds,
    "order_id",
    "left"
).select(
    col("order_id"),
    col("customer_id"),
    to_date(col("order_date")).alias("date"),
    col("city"),
    col("channel"),
    col("item_qty"),
    col("line_revenue"),
    when(col("amount").isNull(), 0.0).otherwise(col("amount")).alias("refund_amount")
)

print(f"Données finales: {final_data.count():,}")

## 7️⃣ Agrégations par (date, ville, canal)

In [ ]:
# 📊 AGRÉGATIONS FINALES
print("📊 Calcul des agrégations...")

result = final_data.groupBy(
    "date", "city", "channel"
).agg(
    spark_count("order_id").alias("orders_count"),
    spark_count("customer_id").alias("unique_customers"),
    spark_sum("item_qty").alias("items_sold"),
    spark_round(spark_sum("line_revenue"), 2).alias("gross_revenue_eur"),
    spark_round(spark_sum("refund_amount"), 2).alias("refunds_eur")
).withColumn(
    "net_revenue_eur", 
    spark_round(col("gross_revenue_eur") - col("refunds_eur"), 2)
).orderBy("date", "city", "channel")

# Cache du résultat pour réutilisation
result = result.cache()

print(f"\n📈 Résultat final: {result.count():,} lignes")

# Aperçu des résultats
print("\n🎯 Aperçu des résultats:")
result.show(10, truncate=False)

## 8️⃣ Performance finale et comparaison

In [ ]:
# ⏱️ MESURE DE PERFORMANCE
total_time = time.time() - start_time

print(f"\n🎯 RÉSULTATS DE PERFORMANCE:")
print(f"⚡ Temps total PySpark: {total_time:.2f}s")
print(f"🐼 Temps Pandas original: ~45s")
print(f"🚀 Amélioration: {45/total_time:.1f}x plus rapide !")

# Statistiques détaillées
print(f"\n📊 STATISTIQUES:")
print(f"- Total lignes résultat: {result.count():,}")
print(f"- Revenue total: {result.agg(spark_sum('gross_revenue_eur')).collect()[0][0]:,.2f}€")
print(f"- Remboursements: {result.agg(spark_sum('refunds_eur')).collect()[0][0]:,.2f}€")
print(f"- Revenue net: {result.agg(spark_sum('net_revenue_eur')).collect()[0][0]:,.2f}€")

# Validation de l'objectif
if total_time < 15:
    print(f"\n✅ OBJECTIF ATTEINT: < 15s ({total_time:.2f}s)")
    print(f"🏆 Score: EXCELLENT")
elif total_time < 25:
    print(f"\n⚠️  Objectif proche: {total_time:.2f}s (cible < 15s)")
    print(f"📈 Score: BON")
else:
    print(f"\n❌ Besoin d'optimisations: {total_time:.2f}s")
    print(f"🔧 Score: À améliorer")

## 9️⃣ Export des résultats

In [ ]:
# 💾 EXPORT EN CSV
print("💾 Export des résultats...")

# Conversion en Pandas pour export CSV (résultat petit)
result_pandas = result.toPandas()

# Format date pour export
result_pandas['date'] = result_pandas['date'].dt.strftime('%Y-%m-%d')

# Export CSV
output_path = "../output/freshkart_pyspark_results.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
result_pandas.to_csv(output_path, sep=';', index=False)

print(f"✅ Résultats exportés: {output_path}")
print(f"📁 Format: CSV avec séparateur ';'")
print(f"📊 Lignes exportées: {len(result_pandas):,}")

## 🔧 Optimisations avancées (Bonus)

In [ ]:
# 📈 ANALYSE DES OPTIMISATIONS
print("🔍 Analyse des optimisations utilisées:")
print("\n✅ Optimisations appliquées:")
print("- 🚀 Chargement pattern matching (31 fichiers → 1 opération)")
print("- 🧠 Cache intelligent sur DataFrames réutilisés")
print("- 📡 Broadcast des petites tables (clients)")
print("- 💥 Explosion JSON native vs boucles Pandas")
print("- ⚡ Lazy evaluation vs chargement complet")
print("- 🎛️ Adaptive Query Execution (AQE)")

print("\n📊 Cache utilization:")
print(f"- clean_customers: {clean_customers.is_cached}")
print(f"- clean_orders: {clean_orders.is_cached}")
print(f"- clean_refunds: {clean_refunds.is_cached}")
print(f"- orders_items: {orders_items.is_cached}")
print(f"- result: {result.is_cached}")

# Explain plan pour analyse avancée
print("\n🔍 Query Plan Analysis:")
result.explain(mode="simple")

## 📋 Validation et tests

In [ ]:
# ✅ TESTS DE VALIDATION
print("✅ Tests de validation:")

# Test 1: Pas de valeurs nulles dans les colonnes clés
null_checks = result.select(
    [spark_count(when(col(c).isNull(), c)).alias(c) for c in result.columns]
).collect()[0].asDict()

print("\n🔍 Vérification valeurs nulles:")
for col_name, null_count in null_checks.items():
    status = "✅" if null_count == 0 else "❌"
    print(f"{status} {col_name}: {null_count} nulls")

# Test 2: Cohérence des montants
print("\n💰 Vérification cohérence montants:")
negative_revenues = result.filter(col("gross_revenue_eur") < 0).count()
negative_net = result.filter(col("net_revenue_eur") < 0).count()

print(f"✅ Revenus bruts négatifs: {negative_revenues}")
print(f"⚠️  Revenus nets négatifs: {negative_net} (normal si refunds > revenue)")

# Test 3: Distribution par canal
print("\n📱 Distribution par canal:")
channel_dist = result.groupBy("channel").agg(
    spark_sum("orders_count").alias("total_orders"),
    spark_sum("gross_revenue_eur").alias("total_revenue")
).collect()

for row in channel_dist:
    print(f"📊 {row.channel}: {row.total_orders:,} commandes, {row.total_revenue:,.2f}€")

print("\n🎯 Validation terminée !")

## 🏁 Conclusions et prochaines étapes

In [ ]:
# 🏁 CONCLUSIONS FINALES
print("🏁 MIGRATION PANDAS → PYSPARK TERMINÉE !")
print("="*50)

print(f"\n📈 GAINS DE PERFORMANCE:")
print(f"⏱️  Temps: {total_time:.2f}s vs 45s Pandas ({45/total_time:.1f}x plus rapide)")
print(f"🧠 Mémoire: Lazy evaluation vs chargement complet")
print(f"🔄 Scalabilité: Prêt pour 10x plus de données")

print(f"\n🛠️  TECHNIQUES MAÎTRISÉES:")
print(f"✅ Pattern matching pour multiples fichiers")
print(f"✅ Explosion JSON native")
print(f"✅ Broadcast joins optimisés")
print(f"✅ Cache intelligent")
print(f"✅ Agrégations distribuées")

print(f"\n🚀 PROCHAINES ÉTAPES:")
print(f"1. 📊 Streaming temps réel (Kafka + Spark Streaming)")
print(f"2. 🤖 Machine Learning (segmentation clients, prédiction churn)")
print(f"3. 🏗️  Pipeline production (Airflow + Docker)")
print(f"4. 📈 Dashboard temps réel (Kafka + Elasticsearch)")

print(f"\n🎉 Félicitations ! Vous maîtrisez maintenant PySpark sur un cas réel !")

# Arrêt propre de Spark
spark.stop()
print("\n🛑 Spark Session fermée proprement")